# Bee1004 — PDF+ analysis

Run-book for **`40xAIR_640neuronlMarker_568pdf_405dapi_Bee1004_PDF561_FULLRES_109z.ome.tif`**.

| | |
|---|---|
| size | 38.3 GB, 109 z-planes |
| shape | `(109, 8871, 19815)` — single channel, PDF 561/568 |
| plane | 8871 × 19815 = **176 megapixels** (a stitched mosaic) |
| voxel | **0.1616 × 0.1616 × 0.6 µm** (from the OME metadata) |

## Read this before running anything

**The file does not fit in memory, and never will.** As float32 it's 76.6 GB and the
pipeline would peak near 766 GB. Your machine has 39 GB. So:

- **2D** runs **plane by plane** — each plane is 703 MB as float32, which is fine. This
  notebook reads pages straight out of the OME-TIFF, so it never writes 38 GB of
  intermediate files.
- **3D** needs the data reduced. Two honest options, and they answer different questions:

| | pixel | fits? | what it's good for |
|---|---|---|---|
| **8× downsample**, whole brain | 1.29 µm | 12 GB ✓ | overview, finding your ROI |
| **2048² crop**, full res | **0.162 µm** | 18 GB ✓ | actually measuring neurons |

> **Use the overview to find the region, then the crop to measure it.** At 8× every fine
> PDF+ process is gone — arborization on that would be fiction.

**A note on the run order below:** each step has a checkpoint where you *look* at
something. Don't skip those. A 45-minute run on a bad threshold is 45 minutes wasted.

## 0. Setup

In [ ]:
import sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

warnings.filterwarnings("ignore")

# Find the HIVE project root by walking up until we see the `hive` package, rather
# than assuming Jupyter was launched from it. Launching from anywhere else would
# otherwise fail with a bare ModuleNotFoundError.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "hive").is_dir() and (_candidate / "prep_stack.py").is_file():
        PROJECT = _candidate
        sys.path.insert(0, str(PROJECT))
        break
else:
    raise SystemExit(
        "Cannot find the HIVE project (a folder containing hive/ and prep_stack.py).\n"
        "Launch Jupyter from ~/Desktop/HIVE_Final, or put this notebook there."
    )
print(f"HIVE project: {PROJECT}")
plt.rcParams.update({"figure.dpi": 110, "image.cmap": "magma"})

# ---- THE FILE ------------------------------------------------------------------
BEE = Path(
    "/Users/deathyang/Library/CloudStorage/GoogleDrive-manuel.giannoni10@gmail.com/"
    ".shortcut-targets-by-id/1gvU96TNioWIr6rG3Vs0qmMP4hLrHQety/Student Projects/"
    "Group 4/Data/Bee Brains/Bee1004_processed/"
    "40xAIR_640neuronlMarker_568pdf_405dapi_Bee1004_PDF561_FULLRES_109z.ome.tif"
)
OUT = Path("bee1004_results"); OUT.mkdir(exist_ok=True)

# ---- from the OME metadata (confirm with the inspect cell below) ----------------
PIXEL_XY = 0.161636973345304     # um
PIXEL_Z  = 0.6                   # um

if not BEE.exists():
    raise SystemExit(f"File not found:\n  {BEE}")
try:
    with tifffile.TiffFile(BEE) as tf:
        _ = tf.series[0].shape
    print(f"OK — file readable\n  {BEE.name}\n  {BEE.stat().st_size/1e9:.1f} GB")
except PermissionError:
    raise SystemExit(
        "PERMISSION DENIED reading the Google Drive folder.\n\n"
        "macOS blocks it by default. Fix:\n"
        "  System Settings > Privacy & Security > Full Disk Access\n"
        "  -> enable Terminal (or your Jupyter launcher), then restart it.\n"
        "Or copy the file to ~/Desktop first."
    )

In [ ]:
import prep_stack

info = prep_stack.inspect(BEE)

**Check the output above.** `channels: 1` confirms this export is PDF only — no channel
to pick. If `voxel size` differs from `PIXEL_XY`/`PIXEL_Z` in the setup cell, fix them there
and re-run: a wrong voxel size silently makes every area and length wrong.

---
# Part 1 — 2D, plane by plane

Each of the 109 planes is analysed independently at **full resolution**. This is where your
quantitative numbers come from: areas, intensities, and (optionally) arborization.

`analyze_stack_planes` reads one page at a time straight from the OME-TIFF — no 38 GB of
intermediates on disk.

In [ ]:
from hive.io_utils import memory_report

PLANE_SHAPE = (info["shape"][1], info["shape"][2])

for clahe in (True, False):
    r = memory_report(PLANE_SHAPE, workers=4, use_clahe=clahe)
    print(f"CLAHE {'ON ' if clahe else 'OFF'} -> {r['peak_bytes']/1e9:5.1f} GB per plane,"
          f"  max workers on this machine: {r['max_workers']}")

print("\nThat is why the run below uses use_clahe=False and WORKERS=4.")
print("Section 9.1 of HIVE_walkthrough.ipynb A/B-tests CLAHE if you want to check it.")

## 1.1 Checkpoint — three planes, then LOOK

Do not run 109 planes on untested parameters. The two that matter:

- **`TOPHAT_RADIUS`** — must be **larger than your biggest PDF+ object**. Too small and
  HIVE eats your signal as background.
- **`MIN_SIZE`** — debris threshold in pixels. Too high deletes real cells.

In [ ]:
from hive.config import ArborConfig, HiveConfig, PreprocessConfig, SegmentationConfig
from hive.pipeline import analyze_stack_planes

# ---- TUNE THESE ----------------------------------------------------------------
TOPHAT_RADIUS = 25      # px; MUST exceed your largest real object
MIN_SIZE      = 200     # px; debris threshold
USE_CLAHE     = False   # see the memory table above
TEST_PLANES   = [50, 54, 58]    # a few planes from the middle of the stack
# --------------------------------------------------------------------------------

def make_cfg(out_folder, arbor=False):
    return HiveConfig(
        output_folder=str(out_folder), mode="2d", channel="gray",
        voxel_size_xy=PIXEL_XY, voxel_unit="um",
        save_intermediate=False,
        preprocess=PreprocessConfig(baseline_method="tophat", tophat_radius=TOPHAT_RADIUS,
                                    gaussian_sigma=1.0, use_clahe=USE_CLAHE),
        segmentation=SegmentationConfig(min_size=MIN_SIZE, threshold_scale=1.0),
        arbor=ArborConfig(enabled=arbor),
    )

t0 = time.perf_counter()
check = analyze_stack_planes(BEE, make_cfg(OUT / "check"), workers=1, planes=TEST_PLANES)
print(f"{len(TEST_PLANES)} planes in {time.perf_counter()-t0:.0f} s "
      f"-> ~{(time.perf_counter()-t0)/len(TEST_PLANES)*109/60:.0f} min for all 109 at 1 worker")
display(check["summary"])

In [ ]:
# LOOK AT IT. This is the single most important cell in the notebook.
from hive.pipeline import analyze_array
from hive.visualization import create_boundary_overlay
from hive.cleanup import normalize_for_display

Z_LOOK = TEST_PLANES[0]
CROP   = (3000, 3800, 8000, 8800)      # a window to inspect; move it around

with tifffile.TiffFile(BEE) as tf:
    plane = tf.series[0].pages[Z_LOOK].asarray()
patch = plane[CROP[0]:CROP[1], CROP[2]:CROP[3]].astype(np.float32)
del plane

res = analyze_array(patch, f"z{Z_LOOK:03d}_patch", make_cfg(OUT / "patch"))
print(f"{res.object_count} objects in this {patch.shape} window — {res.cleanup.summary()}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(normalize_for_display(patch));                axes[0].set_title("raw")
axes[1].imshow(normalize_for_display(res.prepared.measurement))
axes[1].set_title("background-corrected (what gets MEASURED)")
axes[2].imshow(create_boundary_overlay(res.prepared.measurement, res.labels))
axes[2].set_title(f"segmentation — {res.object_count} objects")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

### Are the outlines on your actual PDF+ structures?

- **Objects missed** → lower `MIN_SIZE`, or try `threshold_scale` below 1.0.
- **Background picked up** → raise `MIN_SIZE`, or `threshold_scale` above 1.0.
- **Objects hollow / eaten** → `TOPHAT_RADIUS` is too small. Raise it.
- **Nothing found at all** → check the crop window actually contains cells.

Re-run the two cells above until it looks right. **Then** continue.

In [ ]:
# ---- The full 109-plane run. Expect ~15-25 min. -------------------------------
WORKERS      = 4        # 4 x 7 GB = 28 GB of your 39. Do not raise without checking.
ARBORIZATION = False    # True adds branch + Sholl metrics (slower)
# --------------------------------------------------------------------------------

t0 = time.perf_counter()
result_2d = analyze_stack_planes(BEE, make_cfg(OUT / "2d", arbor=ARBORIZATION),
                                 workers=WORKERS)
print(f"done in {(time.perf_counter()-t0)/60:.1f} min")

m2d, s2d = result_2d["measurements"], result_2d["summary"]
print(f"{len(m2d)} objects across {len(s2d)} planes")
if result_2d["errors"]:
    print("FAILED planes:", result_2d["errors"])
display(m2d.head())

In [ ]:
# Objects and signal through the stack — a quick reality check on the whole run.
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
z = [int(n[1:]) for n in s2d["image"]]
axes[0].plot(z, s2d["object_count"], marker="o", ms=3)
axes[0].set_xlabel("z-plane"); axes[0].set_ylabel("objects"); axes[0].set_title("objects per plane")
axes[1].plot(z, s2d["mean_intensity"], marker="o", ms=3, color="crimson")
axes[1].set_xlabel("z-plane"); axes[1].set_ylabel("mean intensity")
axes[1].set_title("signal vs depth (should decay with depth)")
if len(m2d):
    axes[2].hist(m2d["area_um2"], bins=50)
    axes[2].set_xlabel("object area (um2)"); axes[2].set_ylabel("count")
    axes[2].set_title("size distribution"); axes[2].set_yscale("log")
plt.tight_layout(); plt.show()

print(f"CSV: {(OUT/'2d'/'combined_measurements.csv').resolve()}")

---
# Part 2 — 3D

Two runs, in this order: an **overview** to find the region, then a **full-resolution crop**
to measure it.

## 2.1 Overview — whole brain at 8×

12 GB peak, fits. Pixel becomes 1.29 µm, so this is for **orientation only** — not for
measuring anything fine.

In [ ]:
DS = 8
vol_ds8 = OUT / f"pdf_ds{DS}.tif"

if not vol_ds8.exists():
    t0 = time.perf_counter()
    prep_stack.extract_3d(BEE, channel=0, downsample=DS, out_file=vol_ds8,
                          px_xy=PIXEL_XY, px_z=PIXEL_Z)
    print(f"extracted in {(time.perf_counter()-t0)/60:.1f} min")
else:
    print(f"already exists: {vol_ds8}")

PIXEL_XY_DS = PIXEL_XY * DS          # 8x downsample => 8x the pixel size
print(f"\neffective pixel: {PIXEL_XY_DS:.4f} um  (z unchanged at {PIXEL_Z})")

In [ ]:
from hive.pipeline import analyze_zstack
from hive.config import RenderConfig

cfg3d = HiveConfig(
    input_path=str(vol_ds8), output_folder=str(OUT / "3d_overview"), mode="3d",
    channel="gray",
    voxel_size_xy=PIXEL_XY_DS,      # the DOWNSAMPLED size, not the original
    voxel_size_z=PIXEL_Z, voxel_unit="um",
    preprocess=PreprocessConfig(baseline_method="percentile", baseline_percentile=25,
                                gaussian_sigma=0.8, use_clahe=False),
    segmentation=SegmentationConfig(min_size=200),
    arbor=ArborConfig(enabled=False),        # meaningless at 1.29 um/px
    render=RenderConfig(mode="mip", n_frames=36, colormap="magma", downsample=1),
)
t0 = time.perf_counter()
res3d = analyze_zstack(cfg3d, render_gif=True)
print(f"done in {(time.perf_counter()-t0)/60:.1f} min — {res3d.object_count} objects")
display(res3d.measurements.head())

In [ ]:
# The projection, with FULL-RESOLUTION pixel coordinates on the axes, so you can
# read your ROI straight off the plot.
# a PNG, not a TIFF -- tifffile would choke on the header
import matplotlib.image as mpimg
proj = mpimg.imread(OUT / "3d_overview" / "HIVE_3D_max_projection.png")
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(proj, extent=[0, proj.shape[1]*DS, proj.shape[0]*DS, 0])
ax.set_xlabel("x (full-res pixels)"); ax.set_ylabel("y (full-res pixels)")
ax.set_title("Max projection — read your ROI corners off these axes")
ax.grid(alpha=0.3, color="cyan", lw=0.5)
plt.tight_layout(); plt.show()

from IPython.display import Image as IPyImage, display as ipy_display
gif = res3d.outputs.get("gif")
if gif and Path(gif).exists():
    ipy_display(IPyImage(filename=str(gif)))

## 2.2 The real 3D run — full resolution, cropped to your ROI

Read the corners off the plot above and put them in `ROI` below. Keep it to about
**2048 × 2048** (= 331 × 331 µm of tissue, ~18 GB peak). Larger will not fit.

In [ ]:
# ---- YOUR REGION OF INTEREST (full-res pixel coordinates) ----------------------
ROI = (3000, 5048, 8000, 10048)      # y0, y1, x0, x1
# --------------------------------------------------------------------------------

h, w = ROI[1] - ROI[0], ROI[3] - ROI[2]
r = memory_report((info["shape"][0], h, w), workers=1)
print(r["summary"])
print(f"\ntissue: {h*PIXEL_XY:.0f} x {w*PIXEL_XY:.0f} um at FULL resolution")
if not r["fits"]:
    raise SystemExit("This ROI will not fit. Shrink it (2048x2048 is the practical max).")

In [ ]:
vol_roi = OUT / "pdf_roi_fullres.tif"
if not vol_roi.exists():
    prep_stack.extract_3d(BEE, channel=0, downsample=1, out_file=vol_roi,
                          px_xy=PIXEL_XY, px_z=PIXEL_Z, crop=ROI)
else:
    print(f"already exists: {vol_roi}")

In [ ]:
cfg_roi = HiveConfig(
    input_path=str(vol_roi), output_folder=str(OUT / "3d_roi"), mode="3d", channel="gray",
    voxel_size_xy=PIXEL_XY,         # FULL resolution here — no downsample correction
    voxel_size_z=PIXEL_Z, voxel_unit="um",
    preprocess=PreprocessConfig(baseline_method="tophat",
                                # per-axis: z is 3.7x coarser than xy, so a scalar
                                # radius would over-subtract along z
                                tophat_radius=(2, TOPHAT_RADIUS, TOPHAT_RADIUS),
                                gaussian_sigma=1.0, use_clahe=False),
    segmentation=SegmentationConfig(min_size=MIN_SIZE),
    arbor=ArborConfig(enabled=True, sholl_step=5.0, prune_length=3),
    render=RenderConfig(mode="mip", n_frames=36, colormap="magma", downsample=2),
)
t0 = time.perf_counter()
res_roi = analyze_zstack(cfg_roi, render_gif=True)
print(f"done in {(time.perf_counter()-t0)/60:.1f} min — {res_roi.object_count} objects")
print(res_roi.cleanup.summary())
display(res_roi.measurements.head())

gif = res_roi.outputs.get("gif")
if gif and Path(gif).exists():
    ipy_display(IPyImage(filename=str(gif)))
for kind, path in res_roi.outputs.items():
    print(f"  {kind:16} {path}")

---
# What you have, and what not to do with it

| Output | What it is |
|---|---|
| `bee1004_results/2d/combined_measurements.csv` | one row per object per plane, **full res**, `image` = `z000`…`z108` |
| `bee1004_results/2d/combined_summary.csv` | per-plane counts and mean intensity |
| `bee1004_results/3d_overview/` | 8× whole-brain: mask, mesh, viewer, GIF |
| `bee1004_results/3d_roi/` | full-res ROI: measurements **+ arborization**, mesh, viewer, GIF |

### Three things that would be wrong

**Don't pool the 2D and 3D numbers.** 2D is per-plane at 0.162 µm; the overview is 1.29 µm.
Different scales, different objects, not comparable.

**Don't treat 2D object counts as cell counts.** A cell spanning 12 planes is counted 12
times. Per-plane 2D counts *profiles*, it doesn't census. For a true count you need the 3D
run (which is why the ROI crop exists).

**Don't trust arborization from the overview.** At 1.29 µm/px fine processes are gone. Only
the full-res ROI has the resolution for branch metrics — which is why `arbor` is off in the
overview cell and on in the ROI cell.

### If the segmentation looks wrong

Go back to the checkpoint cell, move `CROP` somewhere else, and re-tune. `--tophat-radius`
and `min_size` do most of the work. Everything downstream inherits whatever you settle on —
and the exact settings are recorded in the Settings sheet of each `HIVE_2D_report.xlsx` /
`HIVE_3D_report.xlsx`, so the run stays reproducible.